# Portfolio Theory with Matrix Algebra — Python

One notebook, start to finish: pull ~400 US stocks (11 GICS sectors) **straight
from Yahoo Finance**, build the log-return matrix, do the matrix-algebra
foundations (correlation, covariance, positive-semidefinite check, portfolio
moments), then optimize the Sharpe ratio and validate it on a train/test split.

Merged from `part1_Matrix_algebra.ipynb` + `part2_Matrix_algebra.ipynb`
(Julia twin: `portfolio_theory_julia.ipynb`).

## 0. Setup

In [ ]:
# pip install -q yfinance pandas numpy scipy plotly pyarrow
import os
import time
import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from scipy.optimize import minimize

START = "2020-01-01"          # sample start (matches the original 2020-2025 window)
CACHE = "prices_2020_latest.parquet"   # local cache so re-runs skip the download
RF_ANNUAL = 0.0               # risk-free rate used in Sharpe ratios below

## 1. Data — straight from Yahoo Finance

No CSVs, no local paths. One curated ticker list (11 GICS sectors), downloaded
in sector chunks with retries, cached to parquet so re-runs are instant.

In [ ]:
SECTORS = {'Communication Services': ['LYV', 'GOOG', 'META', 'DIS', 'NFLX', 'T', 'VZ', 'TMUS', 'CABO', 'NWSA', 'TTWO', 'EA', 'NXST', 'FOX', 'WBD', 'ROKU', 'CHTR', 'OMC', 'ZG', 'SIRI', 'SPOT', 'TCEHY', 'CCOI', 'PINS', 'SNAP', 'BIDU', 'NTES', 'SE', 'BILI', 'CMCSA', 'ZM', 'VOD', 'RCI', 'LUMN', 'AMX', 'SKM', 'NTTYY'], 'Consumer Discretionary': ['AMZN', 'TSLA', 'HD', 'MCD', 'NKE', 'LOW', 'SBUX', 'TJX', 'TGT', 'BKNG', 'GM', 'F', 'ROST', 'MAR', 'LVS', 'ORLY', 'YUM', 'EBAY', 'AZO', 'DHI', 'LEN', 'RCL', 'HLT', 'CMG', 'NVR', 'DRI', 'ULTA', 'DG', 'KMX', 'BBY', 'ETSY', 'HAS', 'GRMN', 'MGM', 'WHR', 'TPR', 'RL', 'GPC', 'WSM', 'BURL', 'POOL', 'CZR', 'WYNN'], 'Consumer Staples': ['PG', 'KO', 'PEP', 'WMT', 'PM', 'MO', 'COST', 'MDLZ', 'UL', 'CL', 'KHC', 'KMB', 'GIS', 'STZ', 'ADM', 'MNST', 'SYY', 'EL', 'TAP', 'CPB', 'CAG', 'HSY', 'TSN', 'MKC', 'CHD', 'KDP', 'CLX', 'LW', 'HRL', 'SJM', 'BG', 'TSCO', 'SFM', 'KR', 'SAM', 'POST', 'INGR', 'FLO'], 'Energy': ['XOM', 'CVX', 'BP', 'EQT', 'COP', 'EOG', 'PBR', 'KMI', 'PSX', 'VLO', 'MPC', 'WMB', 'SU', 'OXY', 'TRGP', 'CTRA', 'APA', 'DVN', 'SLB', 'LNG', 'OVV', 'AR', 'RRC', 'HAL', 'BKR', 'SM', 'MTDR', 'PAA', 'MUR', 'NOG', 'OKE'], 'Financials': ['JPM', 'BAC', 'WFC', 'C', 'AXP', 'BK', 'BLK', 'SCHW', 'USB', 'PNC', 'TFC', 'COF', 'STT', 'CB', 'AIG', 'MET', 'PRU', 'ALL', 'TRV', 'AFL', 'PFG', 'AMP', 'CME', 'ICE', 'SPGI', 'MCO', 'NDAQ', 'V', 'MA', 'PYPL', 'AON', 'MS', 'GS', 'SYF', 'ALLY', 'KEY', 'RF', 'FITB', 'HBAN'], 'Health Care': ['JNJ', 'UNH', 'PFE', 'ABBV', 'LLY', 'MRK', 'TMO', 'ABT', 'DHR', 'BMY', 'AMGN', 'MDT', 'GILD', 'CVS', 'ISRG', 'CNC', 'ELV', 'SYK', 'ZTS', 'REGN', 'VRTX', 'BSX', 'BDX', 'ILMN', 'BIIB', 'EW', 'IQV', 'MCK', 'CI', 'NVO', 'CAH', 'HOLX', 'RMD', 'HCA', 'LH', 'DGX', 'IDXX', 'DVA', 'UHS', 'HUM'], 'Industrials': ['HON', 'UNP', 'UPS', 'RTX', 'CAT', 'BA', 'DE', 'GE', 'MMM', 'LMT', 'FDX', 'NOC', 'GD', 'EMR', 'ITW', 'ETN', 'CSX', 'WM', 'CMI', 'WAB', 'LHX', 'GWW', 'PH', 'ROP', 'TT', 'DAL', 'AAL', 'UAL', 'SWK', 'FAST', 'RSG', 'CTAS', 'XYL', 'IR', 'PNR', 'ROL', 'SNA', 'URI', 'JBHT', 'DOV', 'PWR'], 'Information Technology': ['AAPL', 'MSFT', 'NVDA', 'AVGO', 'ORCL', 'CSCO', 'ACN', 'ADBE', 'TXN', 'IBM', 'CRM', 'AMD', 'INTC', 'QCOM', 'AMAT', 'INTU', 'ADI', 'MU', 'LRCX', 'NOW', 'PAYX', 'ADP', 'CTSH', 'CDNS', 'SNPS', 'KLAC', 'MCHP', 'NXPI', 'ADSK', 'FTNT', 'PANW', 'HPQ', 'KEYS', 'GLW', 'STX', 'WDAY', 'SWKS', 'MPWR', 'ON', 'APH'], 'Materials': ['LIN', 'SHW', 'APD', 'ECL', 'NEM', 'FCX', 'DD', 'VMC', 'MLM', 'LYB', 'PPG', 'DOW', 'NUE', 'IFF', 'ALB', 'EMN', 'MOS', 'FMC', 'AVY', 'CF', 'AA', 'IP', 'PKG', 'BALL', 'HUN', 'RS', 'STLD', 'CLF', 'RPM', 'OLN', 'OC', 'SQM'], 'Real Estate': ['PLD', 'AMT', 'EQIX', 'SPG', 'PSA', 'CCI', 'WELL', 'O', 'DLR', 'EXR', 'AVB', 'EQR', 'ESS', 'VTR', 'CUBE', 'MAA', 'INVH', 'UDR', 'SUI', 'CPT', 'WY', 'KIM', 'FRT', 'REG', 'IRM', 'HST', 'VNO', 'NNN', 'BRX', 'STAG', 'OHI', 'LTC', 'BXP', 'REXR', 'ADC', 'ARE'], 'Utilities': ['NEE', 'SO', 'DUK', 'D', 'AEP', 'EXC', 'SRE', 'XEL', 'PEG', 'ED', 'WEC', 'ES', 'EIX', 'DTE', 'PPL', 'FE', 'AEE', 'CMS', 'ATO', 'NI', 'CNP', 'EVRG', 'LNT', 'PNW', 'AES', 'NRG', 'VST', 'BKH', 'IDA', 'UGI', 'AWK', 'HE']}

TICKERS = [t for s in SECTORS.values() for t in s]
print(f"{len(TICKERS)} tickers across {len(SECTORS)} sectors")

def download_chunk(tickers, start, tries=4):
    """Download adjusted closes for a ticker chunk, retrying on 429s."""
    for a in range(tries):
        try:
            df = yf.download(tickers, start=start, auto_adjust=True,
                             progress=False, threads=True)
            close = df["Close"] if isinstance(df.columns, pd.MultiIndex) else df
            close = close.dropna(axis=1, how="all")
            return close
        except Exception as e:
            print(f"  attempt {a+1} failed ({type(e).__name__}); retrying...")
            time.sleep(5 * (a + 1))
    raise RuntimeError(f"download failed for chunk of {len(tickers)} tickers")

if os.path.exists(CACHE):
    prices = pd.read_parquet(CACHE)
    print("loaded from cache:", prices.shape)
else:
    chunks = []
    for sec, ticks in SECTORS.items():
        print(f"downloading {sec} ({len(ticks)} tickers)...")
        c = download_chunk(ticks, START)
        c.columns = pd.MultiIndex.from_product([[sec], c.columns],
                                               names=["Sector", "Ticker"])
        chunks.append(c)
        time.sleep(2)
    prices = pd.concat(chunks, axis=1).sort_index()
    prices.to_parquet(CACHE)
    print("downloaded:", prices.shape)

# long-format cache for the Julia notebook (Date, Ticker, Close, Sector)
prices.stack(["Sector", "Ticker"], future_stack=True).rename("Close")       .reset_index().to_csv("prices_long.csv", index=False)

# keep tickers with at least 2 years of data and <5% missing
prices = prices.dropna(axis=1, thresh=int(0.95 * len(prices)))
prices = prices.loc[:, prices.count() > 500]
print("after cleaning:", prices.shape)
print("range:", prices.index[0].date(), "->", prices.index[-1].date())

## 2. Log returns and the return matrix

$R_{t,i} = \log(P_{t,i}/P_{t-1,i})$ — dates as rows, tickers as columns.

In [ ]:
# long format (Date, Ticker, Close, Sector), as in the original part 1
long = (prices.stack(["Sector", "Ticker"], future_stack=True)
              .rename("Close").reset_index())

long["LogReturn"] = (long.sort_values(["Ticker", "Date"])
                         .groupby("Ticker")["Close"]
                         .transform(lambda x: np.log(x / x.shift(1))))
long = long.dropna(subset=["LogReturn"]).reset_index(drop=True)

# pivot: the return matrix ret_df (Date x Ticker)
ret_df = (long.pivot(index="Date", columns="Ticker", values="LogReturn")
              .sort_index())
# a few tickers have isolated gaps (halts); drop the affected rows so that
# matrix products below never see NaN
ret_df = ret_df.dropna()
print("return matrix:", ret_df.shape)
print("\nSample of log returns:")
print(ret_df.iloc[:5, :5].round(4).to_string())

## 3. Correlation, covariance, and the PSD check

A correlation matrix must be positive semidefinite — all eigenvalues ≥ 0
(up to numerical noise). This is the matrix-algebra fact the whole notebook
stands on.

In [ ]:
corr_matrix = ret_df.corr()
cov_matrix = ret_df.cov()

eigenvalues = np.linalg.eigvalsh(corr_matrix.values)
is_psd = np.all(eigenvalues >= -1e-8)
print("Is the correlation matrix PSD?", is_psd)
print("Smallest eigenvalue:", eigenvalues.min().round(6))
print("Largest eigenvalue: ", eigenvalues.max().round(3))
print("Condition number:   ", round(eigenvalues.max() / eigenvalues.min(), 1))

## 4. Portfolio metrics — the equal-weighted baseline

For weights $w$: $\mu_p = w^T\mu$, $\sigma_p^2 = w^T\Sigma w$,
$\mathrm{Sharpe}_p = \mu_p/\sigma_p$.

In [ ]:
mu = ret_df.mean()          # per-asset mean log return
sd = ret_df.std()           # per-asset volatility (risk)

W = np.ones(len(ret_df.columns)) / len(ret_df.columns)   # equal weights

port_mean = W @ mu.values
port_var = W @ cov_matrix.values @ W
port_vol = np.sqrt(port_var)

print(f"assets: {len(W)}")
print(f"equal-weight portfolio: mean={port_mean:.6f}, vol={port_vol:.6f}, "
      f"Sharpe={port_mean/port_vol:.4f}")
print("\nTop 5 assets by individual Sharpe (mean/std):")
print((mu / sd).sort_values(ascending=False).head().round(4).to_string())

## 5. Cumulative log returns

In [ ]:
cum = ret_df.cumsum()
fig = go.Figure()
for t in cum.columns:
    fig.add_trace(go.Scatter(x=cum.index, y=cum[t], mode="lines",
                             name=t, line=dict(width=1)))
fig.update_layout(title="Cumulative Log Returns (all tickers)",
                  xaxis_title="Date", yaxis_title="Cumulative log return",
                  width=1000, height=600, showlegend=False)
fig.show()

## 6. Sharpe-ratio optimization

$\max_w \; (w^T\mu - r_f) / \sqrt{w^T\Sigma w}$ s.t. $w \ge 0$, $\sum w = 1$,
with $r_f = 4\%$ annual (daily $r_f/252$, as in the COLAB pipeline).
Two solvers on the same problem: **SLSQP** (explicit $\sum w = 1$ equality
constraint — the clean formulation) and **L-BFGS-B** (box constraints only,
weights renormalized inside the objective; works because Sharpe is
scale-invariant).

In [ ]:
mu_v = ret_df.mean().values          # daily mean log-returns
cov_v = ret_df.cov().values          # daily covariance
RF_DAILY = 0.04 / 252                # risk-free rate, daily (4% annual, as in the COLAB)
n = len(mu_v)
w0 = np.ones(n) / n
bounds = [(0, 1)] * n

def neg_sharpe(w, mu, cov, rf=0.0):
    w = w / w.sum()
    risk = np.sqrt(w @ cov @ w)
    return np.inf if risk == 0 else -((w @ mu) - rf) / risk

# Primary solver: SLSQP with the budget constraint stated explicitly.
res_slsqp = minimize(neg_sharpe, w0, args=(mu_v, cov_v, RF_DAILY), method="SLSQP",
                     bounds=bounds,
                     constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                     options={"maxiter": 300})
w_slsqp = res_slsqp.x / res_slsqp.x.sum()

# Cross-check: L-BFGS-B has no equality constraints, so the budget is enforced
# by renormalizing inside the objective. This works because Sharpe is
# scale-invariant, but the explicit SLSQP formulation above is the clean one.
res_lbfgsb = minimize(neg_sharpe, w0, args=(mu_v, cov_v, RF_DAILY), method="L-BFGS-B",
                      bounds=bounds, options={"maxiter": 300})
w_lbfgsb = res_lbfgsb.x / res_lbfgsb.x.sum()

def sharpe_of(w, rf=0.0):
    return ((w @ mu_v) - rf) / np.sqrt(w @ cov_v @ w)

print(f"SLSQP    in-sample Sharpe: {sharpe_of(w_slsqp, RF_DAILY):.4f} "
      f"(success={res_slsqp.success})")
print(f"L-BFGS-B in-sample Sharpe: {sharpe_of(w_lbfgsb, RF_DAILY):.4f} "
      f"(success={res_lbfgsb.success})")
print(f"Equal-weight in-sample Sharpe: {sharpe_of(w0, RF_DAILY):.4f}")

print("\nTop holdings of the SLSQP portfolio:")
for t, wt in sorted(zip(ret_df.columns, w_slsqp), key=lambda x: -x[1])[:15]:
    if wt > 0.001:
        print(f"  {t}: {wt:.4f}")


## 7. Train/test validation

60% train / 40% test. Optimize on the most volatile assets in-sample, then
score the frozen weights out-of-sample against the equal-weight baseline.

In [ ]:
split = int(len(ret_df) * 0.6)
train, test = ret_df.iloc[:split], ret_df.iloc[split:]
print(f"train: {train.shape}   test: {test.shape}")

# most volatile assets, in-sample (cap at 550 as in the original)
top = train.std().nlargest(min(550, train.shape[1])).index
tr, te = train[top], test[top]

mu_tr, cov_tr = tr.mean().values, tr.cov().values
res = minimize(neg_sharpe, np.ones(len(top)) / len(top),
               args=(mu_tr, cov_tr, RF_DAILY), method="SLSQP",
               bounds=[(0, 1)] * len(top),
               constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
               options={"maxiter": 300})
w_opt = res.x / res.x.sum()
w_eq = np.ones(len(top)) / len(top)

def oos_sharpe(w, df, rf=0.0):
    r = df.values @ w
    return (r.mean() - rf) / r.std()

print(f"Optimized OOS Sharpe:     {oos_sharpe(w_opt, te, RF_DAILY):.4f}")
print(f"Equal-weight OOS Sharpe:  {oos_sharpe(w_eq, te, RF_DAILY):.4f}")

val_opt = np.exp((te.values @ w_opt).cumsum())
val_eq = np.exp((te.values @ w_eq).cumsum())

fig = go.Figure()
fig.add_trace(go.Scatter(x=te.index, y=val_opt, mode="lines",
                         name="Optimized ($1 start)",
                         line=dict(width=2, color="green")))
fig.add_trace(go.Scatter(x=te.index, y=val_eq, mode="lines",
                         name="Equal weight ($1 start)",
                         line=dict(width=2, color="grey", dash="dash")))
fig.update_layout(title="Out-of-sample: optimized vs equal-weight",
                  xaxis_title="Date", yaxis_title="Portfolio value",
                  width=1000, height=600, template="plotly_white")
fig.show()
